# 03. 데이터 구성과 관측 범위

분석 단위는 전국 국제선 출발 시장의 월, 목적지×월, 노선×월이다. 2023–2025년 관측값으로 시장 규모와 목적지별 패턴을 살피고, 2026년 1–7월 관측값은 고정 Test와 7월 기준 예측 입력에 사용한다. 2026년 8–12월은 아직 실적이 없는 예측 구간이다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'scripts'))
import portfolio_data as pdx
engine=pdx.get_engine()
market=pd.read_sql(text('SELECT period_ym,SUM(passengers) AS passengers,SUM(operations) AS operations FROM v_route_monthly WHERE is_international=TRUE GROUP BY period_ym ORDER BY period_ym'),engine)
destination=pd.read_sql(text('SELECT period_ym,destination_airport_std AS destination,SUM(passengers) AS passengers,SUM(operations) AS operations FROM v_route_monthly WHERE is_international=TRUE GROUP BY period_ym,destination_airport_std ORDER BY period_ym,destination'),engine)
holdout=pd.read_sql(text('SELECT * FROM v_route_monthly_2026'),engine)
panel=pd.read_sql(text('SELECT * FROM v_model_destination_monthly_all'),engine)
profile=pd.read_sql(text('SELECT * FROM v_demand_destination_profile'),engine)

## 1. 행 단위·기간·컬럼·결측

In [2]:
frames={'시장 월':market,'목적지×월':destination,'2026 노선×월':holdout,'모델 패널 목적지×월':panel}
catalog=[]
for name,frame in frames.items():
    catalog.append({'자료':name,'행':len(frame),'열':len(frame.columns),
                    '시작':frame.period_ym.min(),'종료':frame.period_ym.max(),
                    '컬럼':', '.join(frame.columns)})
display(pd.DataFrame(catalog))
for name,frame in frames.items():
    print(name,'결측 컬럼');display(frame.isna().sum().loc[lambda x:x>0].to_frame('결측').head())
    display(frame.head())

,자료,행,열,시작,종료,컬럼
0,시장 월,36,3,202301,202512,"period_ym, passengers, operations"
1,목적지×월,6493,4,202301,202512,"period_ym, destination, passengers, operations"
2,2026 노선×월,2072,10,202601,202607,"origin_airport_raw, destination_airport_raw, p..."
3,모델 패널 목적지×월,7825,5,202301,202607,"period_ym, destination_canonical_std, passenge..."


시장 월 결측 컬럼


,결측


,period_ym,passengers,operations
0,202301,2403433.0,13632.0
1,202302,2212589.0,12937.0
2,202303,2256529.0,14407.0
3,202304,2412373.0,15231.0
4,202305,2622755.0,16781.0


목적지×월 결측 컬럼


...

,결측


,origin_airport_raw,destination_airport_raw,period_raw,period_ym,passengers,operations,origin_airport_std,destination_airport_std,route_scope_std,is_international
0,김포(GMP),간사이(KIX),2026년 01월,202601,52407,279,김포,간사이,international,1
1,김포(GMP),간사이(KIX),2026년 02월,202602,47747,280,김포,간사이,international,1
2,김포(GMP),간사이(KIX),2026년 03월,202603,55812,310,김포,간사이,international,1
3,김포(GMP),간사이(KIX),2026년 04월,202604,50346,300,김포,간사이,international,1
4,김포(GMP),간사이(KIX),2026년 05월,202605,52604,310,김포,간사이,international,1


모델 패널 목적지×월 결측 컬럼


,결측


,period_ym,destination_canonical_std,passengers,operations,origin_airport_count
0,202301,간사이,201845.0,1046.0,6
1,202301,과달라하라,0.0,12.0,1
2,202301,괌,41882.0,226.0,2
3,202301,광저우,2343.0,43.0,1
4,202301,구마모도,1785.0,12.0,1


## 2. 기본 분포와 유니버스

In [3]:
display(market[['passengers','operations']].describe().round(1))
display(destination[['passengers','operations']].describe().round(1))
display(holdout[['period_ym','origin_airport_std','destination_airport_std','passengers','operations']].head())
display(profile.groupby('sample_band',dropna=False).agg(destinations=('destination_canonical_std','count'),passengers=('total_passengers','sum')).reset_index())
print('관측 목적지',destination.destination.nunique(),'모델링 유니버스',int(profile.in_modeling_universe.sum()))
assert len(market)==36 and len(destination)==6493 and destination.destination.nunique()==275
assert int(profile.in_modeling_universe.sum())==169
assert market.period_ym.min()=='202301' and market.period_ym.max()=='202512'
assert panel.period_ym.max()=='202607' 

,passengers,operations
count,36.0,36.0
mean,3493720.2,20555.5
std,567233.9,3021.8
min,2212589.0,12937.0
25%,3236007.2,19241.5
50%,3605910.0,21316.0
75%,3848045.0,22545.5
max,4307521.0,24591.0


,passengers,operations
count,6493.0,6493.0
mean,19370.7,114.0
std,39219.6,212.1
min,0.0,0.0
25%,1209.0,14.0
50%,6053.0,39.0
75%,18521.0,114.0
max,362453.0,2001.0


,period_ym,origin_airport_std,destination_airport_std,passengers,operations
0,202601,김포,간사이,52407,279
1,202602,김포,간사이,47747,280
2,202603,김포,간사이,55812,310
3,202604,김포,간사이,50346,300
4,202605,김포,간사이,52604,310


,sample_band,destinations,passengers
0,12_23,29,937481.0
1,24_35,20,1807371.0
2,ge36,149,152048353.0
3,lt12,88,498668.0


관측 목적지 275 모델링 유니버스 169


2023–2025년 관측 목적지 <strong>275개</strong>는 해당 기간 국제선 수요에 한 번이라도 등장한 목적지다. 결합 프로필은 2026년 7월까지의 <strong>286개</strong> 목적지를 포함하고, 이 가운데 관측 월 24개월 이상인 <strong>169개</strong>가 모델링 유니버스다. 275와 169는 서로 다른 기간·분모의 수다. 목적지별 최초 관측월부터 패널 종료까지 내부 빈 월은 피처 생성 시 0으로 채우며, 최초 관측 전을 0 수요로 가정하지 않는다.

## 3. 저장 집계와 월별 합계 대조

In [4]:
saved_market_path=ROOT/'tableau/market_monthly.csv'
saved_dest_path=ROOT/'tableau/destination_monthly.csv'
if saved_market_path.exists() and saved_dest_path.exists():
    saved_market=pd.read_csv(ROOT/'tableau/market_monthly.csv',encoding='utf-8-sig')
    saved_dest=pd.read_csv(ROOT/'tableau/destination_monthly.csv',encoding='utf-8-sig')
    print('저장 CSV',saved_market.shape,saved_dest.shape);display(saved_market.head());display(saved_dest.head())
    assert len(saved_market)==36 and len(saved_dest)==6493
    saved_market['period_ym']=pd.to_datetime(saved_market['기준월']).dt.strftime('%Y%m')
    saved_dest['period_ym']=pd.to_datetime(saved_dest['기준월']).dt.strftime('%Y%m')
    market_match=market.merge(saved_market,on='period_ym',validate='1:1')
    dest_match=destination.merge(saved_dest,left_on=['period_ym','destination'],right_on=['period_ym','목적지_DB'],validate='1:1')
    assert len(market_match)==36 and len(dest_match)==6493
    assert market_match.passengers.eq(market_match['여객수']).all()
    assert dest_match.passengers.eq(dest_match['여객수']).all()
    print('MySQL vs 저장 CSV: 시장 36행·목적지 6,493행 키/여객 수 전행 일치')
else:
    print('저장 Tableau CSV 비교 생략: 파일이 없는 환경')
monthly=destination.groupby('period_ym').passengers.sum().rename('destination_sum').reset_index().merge(market,on='period_ym')
monthly['difference']=monthly.destination_sum-monthly.passengers
assert monthly.difference.eq(0).all()
display(monthly.head())

저장 Tableau CSV 비교 생략: 파일이 없는 환경


,period_ym,destination_sum,passengers,operations,difference
0,202301,2403433.0,2403433.0,13632.0,0.0
1,202302,2212589.0,2212589.0,12937.0,0.0
2,202303,2256529.0,2256529.0,14407.0,0.0
3,202304,2412373.0,2412373.0,15231.0,0.0
4,202305,2622755.0,2622755.0,16781.0,0.0


목적지별 월 합계를 다시 더하면 시장 월 합계와 일치한다. 저장된 집계 CSV가 있는 환경에서는 같은 키·여객 수를 추가로 대조한다. 예측 모델은 이 월간 여객·운항 패널에서 시작한다.